In [ ]:
import os

# Import Colab's secure Secrets utility.
from google.colab import userdata

# Check whether the API key is already available.
if "GOOGLE_API_KEY" not in os.environ:

    # Load the API key from Colab Secrets.
    os.environ["GOOGLE_API_KEY"] = userdata.get(
        "GOOGLE_API_KEY"
    )

# Verify that the key was loaded.
# We do NOT print the actual API key.
print(
    "Gemini API key loaded:",
    bool(os.environ.get("GOOGLE_API_KEY"))
)

Gemini API key loaded: True


In [ ]:
# Import LangChain's Gemini chat model.
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# Create the Gemini chat model.
#
# temperature=0 makes the response more deterministic,
# which is useful for document-based question answering.
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0
)

print("Gemini initialized successfully!")

Gemini initialized successfully!


In [ ]:
# Send a simple test question directly to Gemini.
response = llm.invoke(
    "What is Retrieval-Augmented Generation?"
)

# Print Gemini's generated response.
print(response.text)

**Retrieval-Augmented Generation (RAG)** is a technique in artificial intelligence that improves the accuracy and reliability of Large Language Models (LLMs) by fetching facts from an external knowledge base before generating a response. 

To understand RAG, it helps to use an analogy:
* **Standard LLM (without RAG):** Like a student taking a **closed-book exam**. They must rely solely on what they memorized during training. If they don't know the answer, they might guess or make something up (hallucinate).
* **LLM with RAG:** Like a student taking an **open-book exam**. Before answering a question, they can look up the most relevant information in a library or textbook, and then write a precise, fact-based answer.

---

### Why is RAG Needed?
While LLMs (like GPT-4 or Claude) are incredibly powerful, they have three major limitations:
1. **Knowledge Cutoff:** They only know information up to the date they were last trained.
2. **Hallucinations:** When they don't know an answer, they o

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 1
    }
)

In [ ]:
import fitz

In [ ]:
pdf = fitz.open("company_report.pdf")

In [ ]:
for page_number, page in enumerate(pdf):
    text = page.get_text()

    print(f"\n===== PAGE {page_number + 1} =====\n")
    print(text)


===== PAGE 1 =====

Company Performance Report 2025
The company operates in the technology sector and provides software products to customers across
India. During 2025, the company experienced strong growth in revenue. Product C achieved the
highest revenue among all products. The company plans to expand its product portfolio in 2026.
Product
Revenue 2024
Revenue 2025
Growth
Product A
$100M
$150M
50%
Product B
$200M
$180M
-10%
Product C
$120M
$210M
75%
Product D
$80M
$110M
37.5%
Product C showed the strongest performance during 2025, with revenue increasing significantly
compared with 2024. Management expects continued growth in the next financial year.



In [ ]:
from langchain_core.documents import Document

In [ ]:
documents = []

for page_number, page in enumerate(pdf):

    text = page.get_text()

    if text.strip():
        documents.append(
            Document(
                page_content=text,
                metadata={
                    "source": "company_report.pdf",
                    "page": page_number + 1,
                    "type": "text"
                }
            )
        )

In [ ]:
print(len(documents))

1


In [ ]:
print(documents[0].page_content)

Company Performance Report 2025
The company operates in the technology sector and provides software products to customers across
India. During 2025, the company experienced strong growth in revenue. Product C achieved the
highest revenue among all products. The company plans to expand its product portfolio in 2026.
Product
Revenue 2024
Revenue 2025
Growth
Product A
$100M
$150M
50%
Product B
$200M
$180M
-10%
Product C
$120M
$210M
75%
Product D
$80M
$110M
37.5%
Product C showed the strongest performance during 2025, with revenue increasing significantly
compared with 2024. Management expects continued growth in the next financial year.



In [ ]:
print(documents[0].metadata)

{'source': 'company_report.pdf', 'page': 1, 'type': 'text'}


In [ ]:
for page_number, page in enumerate(pdf):

    tables = page.find_tables()

    print(f"Page {page_number + 1}")
    print("Tables found:", len(tables.tables))

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Page 1
Tables found: 1


In [ ]:
page = pdf[0]

tables = page.find_tables()

table = tables.tables[0]

In [ ]:
table_data = table.extract()

print(table_data)

[['Product', 'Revenue 2024', 'Revenue 2025', 'Growth'], ['Product A', '$100M', '$150M', '50%'], ['Product B', '$200M', '$180M', '-10%'], ['Product C', '$120M', '$210M', '75%'], ['Product D', '$80M', '$110M', '37.5%']]


In [ ]:
for row in table_data:
    print(row)

['Product', 'Revenue 2024', 'Revenue 2025', 'Growth']
['Product A', '$100M', '$150M', '50%']
['Product B', '$200M', '$180M', '-10%']
['Product C', '$120M', '$210M', '75%']
['Product D', '$80M', '$110M', '37.5%']


In [ ]:
headers = table_data[0]

rows = []

for row in table_data[1:]:
    row_dict = dict(zip(headers, row))
    rows.append(row_dict)

rows

[{'Product': 'Product A',
  'Revenue 2024': '$100M',
  'Revenue 2025': '$150M',
  'Growth': '50%'},
 {'Product': 'Product B',
  'Revenue 2024': '$200M',
  'Revenue 2025': '$180M',
  'Growth': '-10%'},
 {'Product': 'Product C',
  'Revenue 2024': '$120M',
  'Revenue 2025': '$210M',
  'Growth': '75%'},
 {'Product': 'Product D',
  'Revenue 2024': '$80M',
  'Revenue 2025': '$110M',
  'Growth': '37.5%'}]

In [ ]:
print(rows[2])

{'Product': 'Product C', 'Revenue 2024': '$120M', 'Revenue 2025': '$210M', 'Growth': '75%'}


In [ ]:
print(rows[2]["Growth"])

75%


In [ ]:
table_documents = []

for row in rows:

    content = (
        f"Product: {row['Product']}. "
        f"Revenue in 2024: {row['Revenue 2024']}. "
        f"Revenue in 2025: {row['Revenue 2025']}. "
        f"Growth: {row['Growth']}."
    )

    table_documents.append(
        Document(
            page_content=content,
            metadata={
                "source": "company_report.pdf",
                "page": 1,
                "type": "table",
                "product": row["Product"]
            }
        )
    )

In [ ]:
for doc in table_documents:
    print(doc.page_content)

Product: Product A. Revenue in 2024: $100M. Revenue in 2025: $150M. Growth: 50%.
Product: Product B. Revenue in 2024: $200M. Revenue in 2025: $180M. Growth: -10%.
Product: Product C. Revenue in 2024: $120M. Revenue in 2025: $210M. Growth: 75%.
Product: Product D. Revenue in 2024: $80M. Revenue in 2025: $110M. Growth: 37.5%.


In [ ]:
for doc in table_documents:
    if doc.metadata["product"] == "Product C":
        print(doc.page_content)

Product: Product C. Revenue in 2024: $120M. Revenue in 2025: $210M. Growth: 75%.


In [ ]:
# Extract txt
from langchain_core.documents import Document

text_documents = []

for page_number, page in enumerate(pdf):

    text = page.get_text()

    if text.strip():

        text_documents.append(
            Document(
                page_content=text,
                metadata={
                    "source": "company_report.pdf",
                    "page": page_number + 1,
                    "type": "text"
                }
            )
        )

print("Text documents:", len(text_documents))

Text documents: 1


In [ ]:
#Tables
table_documents = []

for page_number, page in enumerate(pdf):

    tables = page.find_tables()

    for table_number, table in enumerate(tables.tables):

        table_data = table.extract()

        if not table_data:
            continue

        headers = table_data[0]

        for row in table_data[1:]:

            row_dict = dict(zip(headers, row))

            content = (
                f"Product: {row_dict['Product']}. "
                f"Revenue in 2024: {row_dict['Revenue 2024']}. "
                f"Revenue in 2025: {row_dict['Revenue 2025']}. "
                f"Growth: {row_dict['Growth']}."
            )

            table_documents.append(
                Document(
                    page_content=content,
                    metadata={
                        "source": "company_report.pdf",
                        "page": page_number + 1,
                        "type": "table",
                        "table_number": table_number + 1,
                        "product": row_dict["Product"]
                    }
                )
            )

print("Table documents:", len(table_documents))

Table documents: 4


In [ ]:
!pip install -qU langchain langchain-huggingface sentence-transformers langchain-community langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 18.2 MB/s eta 0:00:00


In [ ]:
#Split the text
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

text_chunks = text_splitter.split_documents(text_documents)

print("Text chunks:", len(text_chunks))

Text chunks: 2


In [ ]:
#Combine
all_documents = text_chunks + table_documents

In [ ]:
print("Total searchable documents:", len(all_documents))

Total searchable documents: 6


In [ ]:
#BGE

from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={
        "normalize_embeddings": True
    },
    query_encode_kwargs={
        "prompt": "Represent this sentence for searching relevant passages: ",
        "normalize_embeddings": True
    }
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vectors = embeddings.embed_documents(
    [doc.page_content for doc in all_documents]
)

In [ ]:
print("Number of vectors:", len(vectors))
print("Vector dimensions:", len(vectors[0]))

Number of vectors: 6
Vector dimensions: 384


In [ ]:
# Import LangChain's FAISS vector store
from langchain_community.vectorstores import FAISS

In [ ]:
!pip install faiss-cpu

# Create a FAISS vector store from our text chunks
# and table documents.
#
# LangChain will:
# 1. Take each Document
# 2. Generate its embedding using BGE
# 3. Store the embeddings in FAISS
# 4. Keep the original Document and metadata
vectorstore = FAISS.from_documents(
    all_documents,
    embeddings
)

print("FAISS vector store created successfully!")

FAISS vector store created successfully!


In [ ]:
# User's question
query = "What was Product C's revenue in 2025?"

# Search FAISS for the 3 most relevant documents
results = vectorstore.similarity_search(
    query,
    k=3
)

# Display the retrieved documents
for i, doc in enumerate(results):

    print(f"\n--- Result {i + 1} ---")

    # Print the actual retrieved content
    print("Content:")
    print(doc.page_content)

    # Print metadata to see whether the result
    # came from text or a table
    print("\nMetadata:")
    print(doc.metadata)


--- Result 1 ---
Content:
Company Performance Report 2025
The company operates in the technology sector and provides software products to customers across
India. During 2025, the company experienced strong growth in revenue. Product C achieved the
highest revenue among all products. The company plans to expand its product portfolio in 2026.
Product
Revenue 2024
Revenue 2025
Growth
Product A
$100M
$150M
50%
Product B
$200M
$180M
-10%
Product C
$120M
$210M
75%
Product D
$80M
$110M
37.5%

Metadata:
{'source': 'company_report.pdf', 'page': 1, 'type': 'text'}

--- Result 2 ---
Content:
Product: Product C. Revenue in 2024: $120M. Revenue in 2025: $210M. Growth: 75%.

Metadata:
{'source': 'company_report.pdf', 'page': 1, 'type': 'table', 'table_number': 1, 'product': 'Product C'}

--- Result 3 ---
Content:
$120M
$210M
75%
Product D
$80M
$110M
37.5%
Product C showed the strongest performance during 2025, with revenue increasing significantly
compared with 2024. Management expects continued gr

In [ ]:
# This question uses different wording
# from the original PDF.
query = "Which product generated the most money in 2025?"

# Retrieve the top 3 relevant documents
results = vectorstore.similarity_search(
    query,
    k=1
)

# Print the retrieved results
for i, doc in enumerate(results):

    print(f"\n--- Result {i + 1} ---")
    print(doc.page_content)
    print("Type:", doc.metadata["type"])


--- Result 1 ---
Product: Product A. Revenue in 2024: $100M. Revenue in 2025: $150M. Growth: 50%.
Type: table


In [ ]:
# Convert the FAISS vector store into a LangChain Retriever.
#
# k=3 means the retriever will return
# the 3 most relevant documents.
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 1
    }
)

print("Retriever created successfully!")

Retriever created successfully!


In [ ]:
# Ask a question through the retriever
query = "What was Product C's revenue in 2025?"

# Retrieve relevant documents
results = retriever.invoke(query)

# Print the retrieved documents
for i, doc in enumerate(results):

    print(f"\n--- Result {i + 1} ---")
    print(doc.page_content)
    print("Metadata:", doc.metadata)


--- Result 1 ---
Company Performance Report 2025
The company operates in the technology sector and provides software products to customers across
India. During 2025, the company experienced strong growth in revenue. Product C achieved the
highest revenue among all products. The company plans to expand its product portfolio in 2026.
Product
Revenue 2024
Revenue 2025
Growth
Product A
$100M
$150M
50%
Product B
$200M
$180M
-10%
Product C
$120M
$210M
75%
Product D
$80M
$110M
37.5%
Metadata: {'source': 'company_report.pdf', 'page': 1, 'type': 'text'}


In [ ]:
def ask_rag(question):
    """
    Ask a question using our current
    FAISS + BGE + Gemini RAG pipeline.
    """

    # -----------------------------
    # 1. Retrieve relevant documents
    # -----------------------------

    results = retriever.invoke(question)


    # -----------------------------
    # 2. Create context
    # -----------------------------

    context = "\n\n".join(
        doc.page_content
        for doc in results
    )


    # -----------------------------
    # 3. Create the prompt
    # -----------------------------

    prompt = f"""
You are a helpful document question-answering assistant.

Answer the question using ONLY the provided context.

Do not use outside knowledge.

If the answer cannot be found in the context,
say:

"I don't know based on the provided document."

Context:
{context}

Question:
{question}

Answer:
"""


    # -----------------------------
    # 4. Generate answer with Gemini
    # -----------------------------

    response = llm.invoke(prompt)


    # -----------------------------
    # 5. Return the answer
    # -----------------------------

    return response.text

In [ ]:
# Ask a question that requires information
# from our PDF table.
answer = ask_rag(
    "What was Product C's revenue in 2025?"
)

print(answer)

Based on the provided document, Product C's revenue in 2025 was $210M.


In [ ]:
# Ask a question that requires information
# from our PDF table.
answer = ask_rag(
    "What was Product C's revenue in 2029?"
)

print(answer)

I don't know based on the provided document.
